In [1]:
import pandas as pd

In [2]:
cubo = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\cubo_1.xlsx", 
                    sheet_name = 'cat4',
                    skiprows=4
                    )

In [3]:
clusters = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\clusters_bdg.xlsx")

In [4]:
clusters_fc = clusters[clusters['UNE']=='FARMACORP']

In [5]:
#base = pd.read_csv("Ventasxfacturas.csv")
#base.to_parquet("Ventasxfacturas.parquet", engine="fastparquet", index=False)

In [5]:
ventasxfact = pd.read_parquet(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\Ventasxfacturas.parquet")

In [6]:
cubo['COD_ARTICULO'] = cubo['COD_ARTICULO'].astype(str)
ventasxfact['COD_ARTICULO'] = ventasxfact['COD_ARTICULO'].astype(str)


In [7]:
cubo_fc = pd.merge(
    cubo,
    clusters_fc,
    left_on='COD_BODEGA',
    right_on='BODEGA'
)

In [8]:
cubo_cat4 = cubo_fc[['COD_ARTICULO', 'CAT 4']].drop_duplicates()

In [9]:
ventasxfact = pd.merge(
  ventasxfact,
  clusters_fc,
  left_on='COD_BODEGA',
  right_on='BODEGA'
 )

In [10]:
# filtering only SKUs on FARMACORP
ventas_farmacia = pd.merge(cubo_cat4,
                            ventasxfact,
                            on='COD_ARTICULO',
                            how='inner'
                            )

### SUPPORT A

In [11]:
ventasxclst = ventas_farmacia.groupby('CLUSTER').agg({'NRO_FACTURAS':'nunique'}).reset_index()

In [12]:
ventasxclst.sort_values(by='NRO_FACTURAS', ascending=False)

,CLUSTER,NRO_FACTURAS
4,MEDIANA-B,961570
5,MEDIANA-C,420084
7,PEQUENA-C,389210
3,MEDIANA-A,336898
1,GRANDE-B,333867
6,PEQUENA-B,246823
9,PERMITIDO-B,225029
13,TRADICIONAL-C,185045
12,TRADICIONAL-B,117388
10,PERMITIDO-C,85879


In [13]:
ventasxcat = ventas_farmacia.groupby(['CAT 4', 'CLUSTER']).agg({'NRO_FACTURAS':'nunique'}).reset_index()

In [15]:
ventasxcat.sort_values(by=['CLUSTER', 'NRO_FACTURAS'], ascending=[True, False])

,CAT 4,CLUSTER,NRO_FACTURAS
1111,ANTIRREUMATICOS NO ESTEROIDEOS,GRANDE-A,7183
160,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,GRANDE-A,5406
753,ANTIGRIPALES EXC.ANTIINFARINGITIS,GRANDE-A,4388
2209,HIPNOTICOS Y SEDANTES,GRANDE-A,2999
2862,OTRAS PREPARACIONES UROLOGICAS,GRANDE-A,2510
...,...,...,...
1926,EDULCORANTES LIQUIDOS,TRADICIONAL-C,1
2017,ESPIRULINA,TRADICIONAL-C,1
2287,HORMONAS DE CRECIMIENTO,TRADICIONAL-C,1
2661,MULTIVITAMINAS PARA NINOS,TRADICIONAL-C,1


In [16]:
ventas_cat_clst = pd.merge(
    ventasxcat,
    ventasxclst,
    on='CLUSTER',
    suffixes=('_CAT4', '_TOTAL_CLST')
)

In [17]:
ventas_cat_clst.sort_values(by=['NRO_FACTURAS_CAT4', 'CLUSTER'], ascending=False)

,CAT 4,CLUSTER,NRO_FACTURAS_CAT4,NRO_FACTURAS_TOTAL_CLST
1115,ANTIRREUMATICOS NO ESTEROIDEOS,MEDIANA-B,140356,961570
164,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,MEDIANA-B,106543,961570
757,ANTIGRIPALES EXC.ANTIINFARINGITIS,MEDIANA-B,99790,961570
1116,ANTIRREUMATICOS NO ESTEROIDEOS,MEDIANA-C,61625,420084
1118,ANTIRREUMATICOS NO ESTEROIDEOS,PEQUENA-C,60804,389210
...,...,...,...,...
1598,CONTRASTE GASTROENTEROGRAFIA,GRANDE-A,1,49677
1885,ECHICANCEA,GRANDE-A,1,49677
2395,INFUSIONES PARA CONTROL DE PESO,GRANDE-A,1,49677
3011,OTROS OFTALMOLOGICO,GRANDE-A,1,49677


In [19]:
ventas_cat_clst['support_cat4'] = ventas_cat_clst['NRO_FACTURAS_CAT4'] / ventas_cat_clst['NRO_FACTURAS_TOTAL_CLST']

In [20]:
ventas_cat_clst = ventas_cat_clst.sort_values(by=['CLUSTER', 'support_cat4'], ascending=[True, False])

### SUPPORT AyB

In [21]:
# selfmerge forfinding pairs of CAT 4 on same invoice
pairs_factura = pd.merge(
    ventas_farmacia,
    ventas_farmacia,
    on=["NRO_FACTURAS", "CLUSTER"],
    suffixes=("_A", "_B")
)


In [22]:
# filtering combinations to avoid duplicates
pairs_factura = pairs_factura[pairs_factura["CAT 4_A"] != pairs_factura["CAT 4_B"]]

# keeping only one order of each pair
pairs_factura = pairs_factura[pairs_factura["CAT 4_A"] > pairs_factura["CAT 4_B"]]


In [23]:
pairs_factura[['CAT 4_A', 'CAT 4_B', 'NRO_FACTURAS', 'CLUSTER']].shape

(3782749, 4)

In [28]:
ventasABxclst = pairs_factura.groupby(['CAT 4_A', 'CAT 4_B', 'CLUSTER']).agg({'NRO_FACTURAS':'nunique'}).reset_index()

In [29]:
ventasABxclst.sort_values(by=['CLUSTER', 'NRO_FACTURAS'], ascending=[True, False])

,CAT 4_A,CAT 4_B,CLUSTER,NRO_FACTURAS
18905,ANTIRREUMATICOS NO ESTEROIDEOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,GRANDE-A,1028
19440,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIGRIPALES EXC.ANTIINFARINGITIS,GRANDE-A,703
8933,ANTIGRIPALES EXC.ANTIINFARINGITIS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,GRANDE-A,543
44962,DESCONGESTIONANTES FARINGITICOS,ANTIGRIPALES EXC.ANTIINFARINGITIS,GRANDE-A,455
19174,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIBACTERIANOS PENICILINAS AMPLIO ESPECTRO,GRANDE-A,429
...,...,...,...,...
172636,YODO,OTRAS VITAMINAS SOLAS Y COMBINADAS,TRADICIONAL-C,1
172688,YODO,POVIDONA,TRADICIONAL-C,1
172694,YODO,PREPARACIONES NASALES SISTEMICAS,TRADICIONAL-C,1
172733,YODO,REACTIVOS PUROS,TRADICIONAL-C,1


In [30]:
ventasxpairs = pd.merge(
    ventasABxclst,
    ventasxclst,
    on='CLUSTER',
    suffixes=('_PAIR', '_TOTAL_CLST')
)

In [32]:
ventasxpairs

,CAT 4_A,CAT 4_B,CLUSTER,NRO_FACTURAS_PAIR,NRO_FACTURAS_TOTAL_CLST
0,AGENTES INMUNOSUPRESORES,AGENTES ANTIREUMATICOS ESPECIFICOS,GRANDE-A,11,49677
1,AGENTES INMUNOSUPRESORES,AGENTES ANTIREUMATICOS ESPECIFICOS,GRANDE-B,24,333867
2,AGENTES INMUNOSUPRESORES,AGENTES ANTIREUMATICOS ESPECIFICOS,GRANDE-C,4,71171
3,AGENTES INMUNOSUPRESORES,AGENTES ANTIREUMATICOS ESPECIFICOS,MEDIANA-A,45,336898
4,AGENTES INMUNOSUPRESORES,AGENTES ANTIREUMATICOS ESPECIFICOS,MEDIANA-B,134,961570
...,...,...,...,...,...
172808,YODO,VITAMINA E SOLA O CON ASOCIACIONES,MEDIANA-B,2,961570
172809,YODO,VITAMINA E SOLA O CON ASOCIACIONES,MEDIANA-C,1,420084
172810,YODO,VITAMINA E SOLA O CON ASOCIACIONES,PEQUENA-B,1,246823
172811,YODO,VITAMINAS A+D ASOCIACIONES SIMPLES,MEDIANA-B,4,961570


In [33]:
ventasxpairs['support_AB'] = (ventasxpairs['NRO_FACTURAS_PAIR'] / ventasxpairs['NRO_FACTURAS_TOTAL_CLST']).round(8)

In [34]:
ventasxpairs.sort_values(by=['CLUSTER', 'NRO_FACTURAS_PAIR'], ascending=[True, False])

,CAT 4_A,CAT 4_B,CLUSTER,NRO_FACTURAS_PAIR,NRO_FACTURAS_TOTAL_CLST,support_AB
18905,ANTIRREUMATICOS NO ESTEROIDEOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,GRANDE-A,1028,49677,0.020694
19440,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIGRIPALES EXC.ANTIINFARINGITIS,GRANDE-A,703,49677,0.014151
8933,ANTIGRIPALES EXC.ANTIINFARINGITIS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,GRANDE-A,543,49677,0.010931
44962,DESCONGESTIONANTES FARINGITICOS,ANTIGRIPALES EXC.ANTIINFARINGITIS,GRANDE-A,455,49677,0.009159
19174,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIBACTERIANOS PENICILINAS AMPLIO ESPECTRO,GRANDE-A,429,49677,0.008636
...,...,...,...,...,...,...
172636,YODO,OTRAS VITAMINAS SOLAS Y COMBINADAS,TRADICIONAL-C,1,185045,0.000005
172688,YODO,POVIDONA,TRADICIONAL-C,1,185045,0.000005
172694,YODO,PREPARACIONES NASALES SISTEMICAS,TRADICIONAL-C,1,185045,0.000005
172733,YODO,REACTIVOS PUROS,TRADICIONAL-C,1,185045,0.000005


In [35]:
ventas_cat_clst_pairs = pd.merge(
    ventas_cat_clst[['CAT 4', 'CLUSTER', 'NRO_FACTURAS_CAT4', 'support_cat4']],
    ventasxpairs,
    left_on=['CAT 4', 'CLUSTER'],
    right_on=['CAT 4_A', 'CLUSTER'],
    #how='left'
)

In [36]:
ventas_pairs = pd.merge(
    ventas_cat_clst_pairs[['CLUSTER', 'NRO_FACTURAS_CAT4', 'support_cat4', 'CAT 4_A','CAT 4_B', 'NRO_FACTURAS_PAIR', 
                            'NRO_FACTURAS_TOTAL_CLST', 'support_AB']],
    ventas_cat_clst[['CAT 4', 'CLUSTER', 'support_cat4', 'NRO_FACTURAS_CAT4']],
    left_on=['CAT 4_B', 'CLUSTER'],
    right_on=['CAT 4', 'CLUSTER'],
    #how='left',
    suffixes=['_A', '_B']
)

In [37]:
ventas_pairs = ventas_pairs[[
    'CLUSTER', 'CAT 4_A', 'CAT 4_B', 
    'NRO_FACTURAS_CAT4_A', 'NRO_FACTURAS_CAT4_B', 'NRO_FACTURAS_PAIR', 'NRO_FACTURAS_TOTAL_CLST', 
    'support_cat4_A', 'support_cat4_B', 'support_AB',
    
]]

In [38]:
ventas_pairs['confidence_AB_A'] = (ventas_pairs['support_AB'] / ventas_pairs['support_cat4_A']).round(8)

In [39]:
ventas_pairs['lift_ABA_B'] = (ventas_pairs['confidence_AB_A'] / ventas_pairs['support_cat4_B']).round(8)

In [40]:
ventas_pairs = ventas_pairs.sort_values(by=['CLUSTER', 'support_cat4_A', 'confidence_AB_A'], ascending=[True,False, False])

In [41]:
ventas_pairs

,CLUSTER,CAT 4_A,CAT 4_B,NRO_FACTURAS_CAT4_A,NRO_FACTURAS_CAT4_B,NRO_FACTURAS_PAIR,NRO_FACTURAS_TOTAL_CLST,support_cat4_A,support_cat4_B,support_AB,confidence_AB_A,lift_ABA_B
7,GRANDE-A,ANTIRREUMATICOS NO ESTEROIDEOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,7183,5406,1028,49677,0.144594,0.108823,0.020694,0.143116,1.315123
42,GRANDE-A,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIGRIPALES EXC.ANTIINFARINGITIS,7183,4388,703,49677,0.144594,0.088331,0.014151,0.097870,1.107996
23,GRANDE-A,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIBACTERIANOS PENICILINAS AMPLIO ESPECTRO,7183,1237,429,49677,0.144594,0.024901,0.008636,0.059724,2.398486
50,GRANDE-A,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIHISTAMINICOS-ANTIALERGICOS,7183,1658,255,49677,0.144594,0.033376,0.005133,0.035500,1.063666
22,GRANDE-A,ANTIRREUMATICOS NO ESTEROIDEOS,ANTIBACTERIANOS MACROLIDOS Y SIMILARES,7183,792,172,49677,0.144594,0.015943,0.003462,0.023945,1.501942
...,...,...,...,...,...,...,...,...,...,...,...,...
172808,TRADICIONAL-C,SUPLEMENTO DE CONTROL DE PESO,ANTIULCEROSOS,3,7157,1,185045,0.000016,0.038677,0.000005,0.333081,8.611845
172809,TRADICIONAL-C,SUPLEMENTO DE CONTROL DE PESO,DIURETICOS,3,1689,1,185045,0.000016,0.009128,0.000005,0.333081,36.491991
172810,TRADICIONAL-C,SUPLEMENTO DE CONTROL DE PESO,OTROS PREP.POLIVIT.MINER,3,1688,1,185045,0.000016,0.009122,0.000005,0.333081,36.513610
172811,TRADICIONAL-C,BARRA DE PROTEINA,ANTIRREUMATICOS NO ESTEROIDEOS,2,28757,1,185045,0.000011,0.155405,0.000005,0.499621,3.214955


In [42]:
ventas_pairs.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\support_analysis_clst.xlsx", index=False)